<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/14_endian/14_endian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Install GNU Fortran and NVIDIA HPC SDK (optional for C-only examples; can take ~30 min)

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download, not needed for C-only examples).
# Uncomment the following lines to install it.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi

import glob, os
nvhpc_bins = sorted(glob.glob('/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin'), key=lambda p: tuple(int(x) for x in p.split('/Linux_x86_64/')[1].split('/')[0].replace('-', '.').split('.')), reverse=True)
if nvhpc_bins:
    nvhpc_bin = nvhpc_bins[0]
    current_path = os.environ.get('PATH', '')
    if nvhpc_bin not in current_path.split(':'):
        os.environ['PATH'] = nvhpc_bin + (':' + current_path if current_path else '')
    print('Using NVIDIA HPC SDK:', nvhpc_bin)
else:
    print('NVIDIA HPC SDK not found (optional; not needed for C-only examples).')


Clone the repository and change to the `14_endian` directory.

In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd HPC-Programming/Tuning/sample_code/14_endian
!ls


# Check endianness  
* Author:   Yukihiro Ota (yota@rist.or.jp)
* Last update: 18th Jan., 2024 

## Instruction: Compile
1. Source code is stored in `src/`. Choose either fortran or c.
2. Change directory

In [ ]:
!cd src/c                   # C
!cd src/fortran/native      # Fortran, with system's native setting of endianness
!cd src/fortran/swap        # Fortran, with the swapped version


3. Make

In [ ]:
!make


The code is successfully compiled by GNU compiler (8.5.0).  
  * Remark: In fortran, you need to consider two patterns. 

## Instruction: Run and do analyses
1. Sample scripts are stored in `tests/`. Choose either fortran or c.       
2. Change directory

In [ ]:
!cd tests/c # On c


3. Run a job script, `run.sh`.

In [ ]:
# One example
!bash run.sh
# Another example
!chmod 755 run.sh
!./run.sh


4. The results will be summarized in file, `logfile`

## Description
You may desire to save your results as a binary file, rather that a text (ASCII) file. In this case, you have to take care of endianness in your computer. Endianness means in what kinds of byte order data (integer and 
floating-point) is arranged. Two kinds of endianness are typically used.
  * little endian (examples: Intel x86, ARM)
  * big endian (examples: SPARC, IBM, Cray)  

In a Linux system, you can check a kind of endianness using `lscpu`. 

Let us consider an unsigned integer with 16 bit, for example, `0x1234`. This is a hexadecimal number. Notice that `34` consists of data with 1 byte (`= 4 bits *2 = 8 bits = 1 byte`). The byte order means the order of arranging upper byte (`12`) and lower bytes (`34``).

In [ ]:
!Byte address:        0   1
!little endian:      34  12
!big endian:         12  34


From a performance point of view the use of binary data in file I/O seems to be attractive, but we should recognize several disadvantages. See, e.g., Richard Gerber's presentation in High Performance Parallel I/O  (August 6, 2013) [http://www.nersc.gov/about/nersc-staff/center-leadership/richard-gerber](http://www.nersc.gov/about/nersc-staff/center-leadership/richard-gerber). 

## Exercise
1. Check the endianness of your system.